In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

import optuna

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

import joblib
import matplotlib.pyplot as plt
import seaborn as sns

c:\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = pd.read_csv('Leads_Selected_Train.csv')
test_df = pd.read_csv('Leads_Selected_Test.csv')

TARGET = 'Converted'

In [3]:
full = pd.concat([train_df, test_df], axis=0)

for col in full.select_dtypes(include='object').columns:
    le = LabelEncoder()
    full[col] = le.fit_transform(full[col].astype(str))

train_df = full.iloc[:len(train_df)]
test_df = full.iloc[len(train_df):]

X_train = train_df.drop(TARGET, axis=1)
y_train = train_df[TARGET]

X_test = test_df.drop(TARGET, axis=1)
y_test = test_df[TARGET]

In [4]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "use_label_encoder": False,
        "eval_metric": "logloss"
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=30)

best_xgb = XGBClassifier(**study_xgb.best_params)
best_xgb.fit(X_train, y_train)

[I 2026-05-02 18:01:36,407] A new study created in memory with name: no-name-8e4b7dac-8d1a-444f-809c-40bdbf6013ec
c:\Python314\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:01:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2026-05-02 18:01:38,514] Trial 0 finished with value: 0.7984395318595578 and parameters: {'n_estimators': 413, 'max_depth': 8, 'learning_rate': 0.224902154218076, 'subsample': 0.8582589035351511, 'colsample_bytree': 0.6829507645125981}. Best is trial 0 with value: 0.7984395318595578.
c:\Python314\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:01:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2026-05-02 18:01:38,568] Trial 1 finished with value: 0.8120936280884266 and parameters: {'n_estimators':

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8706799555456454
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegres

In [5]:
def objective_lgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0)
    }

    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)

study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb, n_trials=30)

best_lgb = LGBMClassifier(**study_lgb.best_params)
best_lgb.fit(X_train, y_train)

[I 2026-05-02 18:01:55,798] A new study created in memory with name: no-name-3ea697e3-eeb5-4db5-ae90-bc04283bac25


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 2453, number of negative: 3699
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000227 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 353
[LightGBM] [Info] Number of data points in the train set: 6152, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.398732 -> initscore=-0.410751
[LightGBM] [Info] Start training from score -0.410751
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[I 2026-05-02 18:01:58,368] Trial 0 finished with value: 0.7945383615084526 and parameters: {'n_estimators': 237, 'num_leaves': 140, 'learning_rate': 0.13960010915146967, 'subsample': 0.828911042031471, 'colsample_bytree': 0.7606385323332314}. Best is trial 0 with value: 0.7945383615084526.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 2453, number of negative: 3699
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000045 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 353
[LightGBM] [Info] Number of data points in the train set: 6152, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.398732 -> initscore=-0.410751
[LightGBM] [Info] Start training from score -0.410751
[LightGBM] [Warning] No further splits with positive gain, best gai

[I 2026-05-02 18:01:58,578] Trial 1 finished with value: 0.8101430429128739 and parameters: {'n_estimators': 204, 'num_leaves': 76, 'learning_rate': 0.06980788536633759, 'subsample': 0.9011680972089542, 'colsample_bytree': 0.6847023077285435}. Best is trial 1 with value: 0.8101430429128739.


...

[I 2026-05-02 18:02:07,339] Trial 28 finished with value: 0.7971391417425228 and parameters: {'n_estimators': 326, 'num_leaves': 150, 'learning_rate': 0.037980600320381426, 'subsample': 0.8326446218444502, 'colsample_bytree': 0.8673588025927687}. Best is trial 23 with value: 0.8179453836150845.


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 2453, number of negative: 3699
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000046 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 353
[LightGBM] [Info] Number of data points in the train set: 6152, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.398732 -> initscore=-0.410751
[LightGBM] [Info] Start training from score -0.410751


[I 2026-05-02 18:02:07,607] Trial 29 finished with value: 0.7873862158647594 and parameters: {'n_estimators': 221, 'num_leaves': 102, 'learning_rate': 0.22837622225420046, 'subsample': 0.8121692996955462, 'colsample_bytree': 0.9178583289229939}. Best is trial 23 with value: 0.8179453836150845.


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 2453, number of negative: 3699
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000048 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 353
[LightGBM] [Info] Number of data points in the train set: 6152, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.398732 -> initscore=-0.410751
[LightGBM] [Info] Start training from score -0.410751


,boosting_type,'gbdt'
,num_leaves,108
,max_depth,-1
,learning_rate,0.01060465973887723
,n_estimators,310
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [6]:
def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 5, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5)
    }

    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)

study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(objective_rf, n_trials=30)

best_rf = RandomForestClassifier(**study_rf.best_params)
best_rf.fit(X_train, y_train)

[I 2026-05-02 18:02:14,701] A new study created in memory with name: no-name-2d35af53-f050-41c4-94ee-16bc2c2e4575
[I 2026-05-02 18:02:16,336] Trial 0 finished with value: 0.823146944083225 and parameters: {'n_estimators': 459, 'max_depth': 19, 'min_samples_split': 2, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.823146944083225.
[I 2026-05-02 18:02:17,672] Trial 1 finished with value: 0.817295188556567 and parameters: {'n_estimators': 354, 'max_depth': 16, 'min_samples_split': 7, 'min_samples_leaf': 3}. Best is trial 0 with value: 0.823146944083225.
[I 2026-05-02 18:02:18,745] Trial 2 finished with value: 0.8302990897269181 and parameters: {'n_estimators': 344, 'max_depth': 10, 'min_samples_split': 4, 'min_samples_leaf': 5}. Best is trial 2 with value: 0.8302990897269181.
[I 2026-05-02 18:02:19,682] Trial 3 finished with value: 0.8296488946684005 and parameters: {'n_estimators': 341, 'max_depth': 8, 'min_samples_split': 5, 'min_samples_leaf': 1}. Best is trial 2 with value: 0.8

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",263
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",9
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",6
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [7]:
import os
os.makedirs("results", exist_ok=True)

def save_confusion_matrix(model, name):
    preds = model.predict(X_test)
    cm = confusion_matrix(y_test, preds)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.savefig(f"results/cm_{name}.png")
    plt.close()

save_confusion_matrix(best_xgb, "xgb")
save_confusion_matrix(best_lgb, "lgb")
save_confusion_matrix(best_rf, "rf")

In [8]:
joblib.dump(best_xgb, "results/xgb.pkl")
joblib.dump(best_lgb, "results/lgb.pkl")
joblib.dump(best_rf, "results/rf.pkl")

['results/rf.pkl']